# OPTW

In the orienteering problem with time windows (OPTW), we are given a set of locations $N = \{ 0, ..., n+1 \}$.
The vehicle starts from the depot $0$, visits each customer $i \in \{ 1, ..., n \}$ at most once, and must reach the goal $n+1$ by the deadline $b_{n+1}$.
By visiting customer $i$, profit $p_i$ is obtained.
The traveling time from $i$ to $j$ is $c_{ij}$.
Each customer $i$ must be visited within time window $[a_i, b_i]$, and the vehicle must wait until $a_i$ if arriving at $i$ before $a_i$.
The objective is to maximize the total profit.

## DP Formulation

Let $R \subseteq N \setminus \{ 0, n+1 \}$ be the set of reachable customers, $i \in N$ be the current location of the vehicle, and $t$ be the current time.
Let $V(R, i, t)$ be the maximum additional profit that can be achieved from state $(R, i, t)$.
When customer $j \in R$ is visited next, the current location is updated to $j$, and the current time is updated to $t_j = \max \{ t + c_{ij}, a_j \}$, the arrival time at $j$ after waiting until $a_j$ if the vehicle arrives early.
The set of reachable customers is updated to $\{ k \in R \setminus \{ j \} \mid t_j + c_{jk} \leq b_k \}$, assuming the triangle inequality for the travel time.
The vehicle may also choose to finish by heading directly to the goal, which is possible only if it can arrive by the deadline, i.e., $t_{n+1} = t + c_{i,n+1} \leq b_{n+1}$; finishing collects no further profit.

$$
\begin{align}
    \text{compute } & V(N \setminus \{ 0, n+1 \}, 0, 0) \\
    & V(R, i, t) = \begin{cases}
         \max_{j \in R \cup \{ n + 1 \}, t + c_{ij} \leq b_j} p_{j} + V(R_j, j, t_j) & \text{if } i \neq n + 1 \\
         0 & \text{if } i = n+1,
    \end{cases}
\end{align}
$$

where $R_j = \{ k \in R \setminus \{ j \} \mid t_j + c_{jk} \leq b_k \}$ for $j \in N \setminus \{ 0, n + 1 \}$ and $R_{n+1} = \emptyset$.
We treat the goal as an additional customer with $p_{n+1} = 0$ and $a_{n+1} = 0$, so that finishing collects no further profit and requires no waiting -- this is what lets the single case above also cover heading directly to the goal.
If no $j \in R \cup \{ n + 1 \}$ satisfies $t + c_{ij} \leq b_j$, the maximum is over an empty set, which we take to be $-\infty$: by the triangle inequality, once the goal is unreachable directly it can never become reachable through a detour, so such a state has no valid continuation.
The second case reflects that, once the vehicle has reached the goal, no more profit can be collected.

When two states $(R, i, t)$ and $(R', i, t')$ have the same location $i$ and $t \leq t' \land R' \subseteq R$, $(R, i, t)$ leads to a better solution. Therefore,

$$
    V(R, i, t) \geq V(R', i, t') \text{ if } t \leq t' \land R' \subseteq R.
$$

The total profit can be overestimated by solving the fractional knapsack problem, where each item $j \in R$ is a customer, its value is the profit $p_j$, and its weight is the minimum possible travel time to reach $j$, $\mathsf{min\_to}_j = \min_{k \in N \setminus \{ j, n+1 \}} c_{kj}$ (precomputed, excluding the goal since it cannot be an intermediate stop), and the knapsack has the capacity of $b_{n+1} - t - \mathsf{min\_to}_{n+1}$.

$$
    V(R, i, t) \leq \left\lfloor \mathsf{fractional\_knapsack}\left(R, b_{n+1} - t - \mathsf{min\_to}_{n+1}, (p_j)_{j \in R}, \left(\mathsf{min\_to}_j \right)_{j \in R} \right) \right\rfloor.
$$

Similarly, we can use the minimum travel time to leave each customer, $\mathsf{min\_from} = \min_{k \in N \setminus \{ j, n+1 \}} c_{jk}$, as the weight, and reserve the time to leave the current location instead:

$$
    V(R, i, t) \leq \left\lfloor \mathsf{fractional\_knapsack}\left(R, b_{n+1} - t - \mathsf{min\_from}_i, (p_j)_{j \in R}, \left(\mathsf{min\_from}_j \right)_{j \in R} \right) \right\rfloor.
$$

## Install DIDPPy

In [1]:
!pip install didppy

In [2]:
# Number of customers
n = 3
# Profit
p = [0, 2, 3, 4, 0]
# Ready time
a = [0, 5, 0, 8, 0]
# Due time
b = [100, 12, 10, 12, 17]
# Travel time
c = [
    [0, 3, 4, 5, 0],
    [3, 0, 5, 4, 3],
    [4, 5, 0, 3, 4],
    [5, 4, 3, 0, 5],
    [0, 3, 4, 5, 0],
]

In [3]:
import math

import didppy as dp

model = dp.Model(maximize=True)

customer = model.add_object_type(number=n + 2)

reachable = model.add_set_resource_var(object_type=customer, target=list(range(1, n + 1)), less_is_better=False)
location = model.add_element_var(object_type=customer, target=0)
time = model.add_int_resource_var(target=0, less_is_better=True)

travel_time = model.add_int_table(c)
due_time = model.add_int_table(b)

k = model.add_local_var()

for j in range(1, n + 1):
    time_next = dp.max(time + travel_time[location, j], a[j])
    visit = dp.Transition(
        name="visit {}".format(j),
        cost=p[j] + dp.IntExpr.state_cost(),
        preconditions=[
            reachable.contains(j),
            time + travel_time[location, j] <= due_time[j],
        ],
        effects=[
            (reachable, reachable.remove(j).filter(k, time_next + travel_time[j, k] <= due_time[k])),
            (location, j),
            (time, time_next),
        ],
    )
    model.add_transition(visit)

empty_set = model.create_set_const(object_type=customer, value=[])
finish = dp.Transition(
    name="finish",
    cost=dp.IntExpr.state_cost(),
    effects=[(reachable, empty_set), (location, n + 1), (time, time + travel_time[location, n + 1])],
    preconditions=[time + travel_time[location, n + 1] <= due_time[n + 1]],
)
model.add_transition(finish)

model.add_base_case([location == n + 1])

profit = model.add_int_table(p)

min_to = model.add_int_table([min(c[i][j] for i in range(n + 1) if i != j) for j in range(n + 2)])

model.add_dual_bound(math.floor(dp.fractional_knapsack(reachable, due_time[n + 1] - time - min_to[n + 1], profit, min_to)))

min_from = model.add_int_table([min(c[j][i] for i in range(n + 1) if i != j) for j in range(n + 2)])

model.add_dual_bound(math.floor(dp.fractional_knapsack(reachable, due_time[n + 1] - time - min_from[location], profit, min_from)))

## Solving

In [4]:
solver = dp.CABS(model, quiet=True)
solution = solver.search()

print("Transitions to apply:")
print("")

for t in solution.transitions:
    print(t.name)

print("")
print("Cost: {}".format(solution.cost))

Transitions to apply:

visit 2
visit 3
visit 1
finish

Cost: 9
